In [1]:
!pip install -U diffusers

In [2]:
!pip install transformers[sentencepiece]

In [5]:
#
# Log in using your Hugging Face API tokengit 


In [2]:
import torch
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image, make_image_grid

# Load a stable diffusion img2img pipeline (CPU mode)
pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float32,   # use float32 on CPU
    use_safetensors=True
)

# Enable CPU memory optimizations
pipe.enable_sequential_cpu_offload()

# Load an input image
init_image = load_image("your_input_image_path_or_url")

# Run the pipeline
prompt = "A fantasy creature in painterly style"
result = pipe(prompt, image=init_image, strength=0.7, guidance_scale=7.5).images[0]

# Show both input and output
make_image_grid([init_image, result], rows=1, cols=2)


RuntimeError: Failed to import diffusers.pipelines.auto_pipeline because of the following error (look up to see its traceback):
cannot import name 'AutoImageProcessor' from 'transformers' (/mnt/abka03/.conda/envs/xl_vlm/lib/python3.9/site-packages/transformers/__init__.py)

In [6]:
from diffusers import FluxPipeline
import torch

cache_dir = '/mnt/abka03/huggingface/hub'

model_name = "black-forest-labs/FLUX.1-dev"

# Load pipeline with float16 for lower memory usage
pipe = FluxPipeline.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    torch_dtype=torch.float16, # Use float16 for GPU
    # revision="fp16" # Uncomment if model repo supports fp16 branch
 )
pipe.to("cuda")  # Move model to GPU

prompt = "A cat holding a sign that says hello world"
image = pipe(
    prompt,
    guidance_scale=0.0,
    output_type="pil",
    num_inference_steps=4,
    max_sequence_length=256,
    generator=torch.Generator("cuda").manual_seed(0)
 ).images[0]
image.save("flux-schnell.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 